In [6]:
import os
import requests
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine
from dotenv import load_dotenv

# Batch Data Load 2020 - 2026

In [8]:
# URL Archive Jakarta Pusat (2020 - sekarang)
url = "https://archive-api.open-meteo.com/v1/archive?latitude=-6.1823&longitude=106.8293&start_date=2020-01-01&end_date=2026-02-01&hourly=temperature_2m,relative_humidity_2m,precipitation,surface_pressure,wind_speed_10m,weather_code&timezone=Asia%2FBangkok"

response = requests.get(url)

if response.status_code == 200:
    hourly = response.json()['hourly']
    
    # Mapping langsung dengan nama kolom yang baru/bagus
    df = pd.DataFrame({
        'time_interval': pd.to_datetime(hourly['time']), # Langsung diconvert ke datetime
        'temperature': hourly['temperature_2m'],
        'humidity': hourly['relative_humidity_2m'],
        'precipitation': hourly['precipitation'],
        'pressure': hourly['surface_pressure'],
        'wind_speed': hourly['wind_speed_10m'],
        'weather_code': hourly['weather_code']
    })
    
    # Tambahkan timestamp ekstraksi
    df['extracted_at'] = datetime.now()
    
    print(f"Data siap! Total: {len(df)} baris.")
else:
    print(f"Gagal! Status code: {response.status_code}")
    

Data siap! Total: 53376 baris.


In [9]:
load_dotenv() 
DATABASE_URL = os.getenv('DATABASE_URL')

engine = create_engine(DATABASE_URL)

try:
    # Mengirim data ke tabel 'weather_history'
    df.to_sql('weather_history', engine, if_exists='append', index=False)
    print("Berhasil! 53.000+ data sudah masuk ke Neon.")
except Exception as e:
    print(f"Terjadi kesalahan: {e}")

Berhasil! 53.000+ data sudah masuk ke Neon.
